# Air Quality Prediction and Pollution Episode Detection

**Problem statement:** predict a selected pollutant using historical pollutant readings and environmental variables,
engineer lagged features, evaluate on future periods, flag periods where observations differ substantially from the
model, and analyze whether the flagged episodes are unusual pollution or data limitations.

**On Kaggle:** add the UCI Air Quality dataset as input (or switch Internet on), optionally turn on a GPU, then *Run All*.

## Setup

In [ ]:
import glob, io, os
import json
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# Pollutants the user can select (NMHC is left out: 90% of its values are missing).
POLLUTANTS = {
    "CO":   {"column": "CO(GT)",   "unit": "mg/m³", "name": "Carbon monoxide"},
    "NO2":  {"column": "NO2(GT)",  "unit": "µg/m³", "name": "Nitrogen dioxide"},
    "NOx":  {"column": "NOx(GT)",  "unit": "ppb",   "name": "Nitrogen oxides"},
    "C6H6": {"column": "C6H6(GT)", "unit": "µg/m³", "name": "Benzene"},
}
# Model inputs = the fields of POST /predict. Everything is known before the hour being predicted.
FEATURES = ["hour", "weekday", "lag_1", "lag_2", "lag_3", "lag_24",
            "temperature", "relative_humidity", "absolute_humidity"]

## 1. Load and clean the data
`-200` means missing. Rows are placed on a continuous hourly timeline.

In [ ]:
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
MODELS = os.path.join(OUT, "models")
os.makedirs(MODELS, exist_ok=True)
RESULTS, pred_frames = {}, {}


def read_table(source):
    """Original UCI format (';' and ',' decimals) or a comma-separated copy."""
    for kwargs in ({"sep": ";", "decimal": ","}, {}):
        if hasattr(source, "seek"):
            source.seek(0)
        t = pd.read_csv(source, **kwargs)
        t.columns = t.columns.str.strip()
        if "CO(GT)" in t.columns:
            return t
    raise ValueError("No CO(GT) column found")


def load_data():
    files = [f for root in ["/kaggle/input", "."] for f in glob.glob(os.path.join(root, "**", "*"), recursive=True)
             if f.lower().endswith((".csv", ".zip")) and "air" in os.path.basename(f).lower()]
    raw = None
    for f in sorted(files, key=lambda f: f.endswith(".zip")):
        try:
            if f.endswith(".zip"):
                z = zipfile.ZipFile(f)
                raw = read_table(io.BytesIO(z.read(next(n for n in z.namelist() if n.lower().endswith(".csv")))))
            else:
                raw = read_table(f)
            print("Using", f)
            break
        except Exception as exc:
            print("Skipping", f, exc)
    if raw is None:
        import urllib.request
        print("Downloading from UCI ...")
        data = urllib.request.urlopen("https://archive.ics.uci.edu/static/public/360/air+quality.zip").read()
        raw = read_table(io.BytesIO(zipfile.ZipFile(io.BytesIO(data)).read("AirQualityUCI.csv")))
    raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]
    if {"Date", "Time"} <= set(raw.columns):
        raw = raw.dropna(subset=["Date", "Time"])
        time = pd.to_datetime(raw["Date"].astype(str) + " " + raw["Time"].astype(str).str.replace(".", ":", regex=False),
                              dayfirst=True)
        df = raw.drop(columns=["Date", "Time"])
    else:
        tcol = next(c for c in raw.columns if c.lower() in ("datetime", "date_time", "timestamp"))
        time = pd.to_datetime(raw[tcol])
        df = raw.drop(columns=[tcol])
    df = df.apply(pd.to_numeric, errors="coerce").replace(-200, np.nan).dropna(axis=1, how="all")
    df.index = pd.DatetimeIndex(time, name="timestamp")
    df = df[~df.index.duplicated()].sort_index().asfreq("h")
    if df["CO(GT)"].notna().all():
        print("WARNING: no missing values at all - this copy was probably pre-filled; results will not match the original data.")
    return df


df = load_data()
print(f"{len(df)} hourly rows: {df.index.min()} -> {df.index.max()}")
(df.isna().mean() * 100).round(1).to_frame("missing %")

## 2. Quick look at the data

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
for key, info in POLLUTANTS.items():
    s = df[info["column"]]
    (s / s.mean()).resample("D").mean().plot(ax=ax[0], label=key, lw=0.8)
    (s / s.mean()).groupby(df.index.hour).mean().plot(ax=ax[1], label=key, marker="o")
ax[0].set_title("Daily average (relative to each pollutant's mean)"); ax[0].legend()
ax[1].set_title("Average by hour of day (relative)"); ax[1].set_xlabel("hour"); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Lagged features
To predict hour *t*: the pollutant at *t-1, t-2, t-3, t-24* and the weather at *t-1*.

In [ ]:
# 2. Lagged features ------------------------------------------------------------------------
def make_features(df, column):
    """Inputs for predicting hour t: pollutant at t-1, t-2, t-3, t-24 and weather at t-1."""
    s = df[column]
    X = pd.DataFrame(index=df.index)
    X["hour"] = df.index.hour
    X["weekday"] = df.index.day_of_week
    for k in (1, 2, 3, 24):
        X[f"lag_{k}"] = s.shift(k)
    X["temperature"] = df["T"].shift(1)
    X["relative_humidity"] = df["RH"].shift(1)
    X["absolute_humidity"] = df["AH"].shift(1)
    return X[FEATURES], s


def metrics(y_true, y_pred):
    return {"MAE": float(mean_absolute_error(y_true, y_pred)),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "R2": float(r2_score(y_true, y_pred)), "n": int(len(y_true))}


def gpu_available():
    try:
        XGBRegressor(n_estimators=2, device="cuda").fit(np.random.rand(20, 2), np.random.rand(20))
        return True
    except Exception:
        return False


DEVICE = "cuda" if gpu_available() else "cpu"

## 4. Models and tuning grids
Linear Regression, Random Forest and XGBoost (GPU if available), compared with a persistence baseline.

In [ ]:
# 3. Candidate models and their tuning grids ------------------------------------------------
CANDIDATES = {
    "Linear Regression": (lambda p: LinearRegression(), [{}]),
    "Random Forest": (lambda p: RandomForestRegressor(random_state=42, n_jobs=2, **p),
                      list(ParameterGrid({"n_estimators": [200], "max_depth": [8, 12, None],
                                          "min_samples_leaf": [1, 5]}))),
    "XGBoost": (lambda p: XGBRegressor(random_state=42, device=DEVICE, tree_method="hist", verbosity=0, **p),
                list(ParameterGrid({"n_estimators": [300, 600], "max_depth": [3, 5],
                                    "learning_rate": [0.05, 0.1]}))),
}


def pipeline(estimator):
    # Missing inputs (gaps in the data) are filled with the training median.
    return Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler()),
                     ("model", estimator)])

## 5. Train, evaluate, flag and analyze
Chronological split: **train 60% → validation 20% → test 20%**. Each model is tuned on validation, the best one is
retrained on train + validation and tested once. Test hours whose error exceeds the 95th percentile of validation errors
are flagged and grouped into episodes, then classified as *data limitation* (missing inputs or a recent data gap) or
*unusual pollution*.

In [ ]:
def train_pollutant(df, key):
    info = POLLUTANTS[key]
    X, y = make_features(df, info["column"])
    n = len(df)
    train_end, val_end = int(n * 0.6), int(n * 0.8)

    # Score only hours where the true value and the previous hour exist (so persistence is defined).
    usable = (y.notna() & X["lag_1"].notna()).to_numpy()
    rows = np.arange(24, n)
    train = rows[(rows < train_end) & y.notna().to_numpy()[rows]]
    val = rows[(rows >= train_end) & (rows < val_end) & usable[rows]]
    test = rows[(rows >= val_end) & usable[rows]]
    train_val = rows[(rows < val_end) & y.notna().to_numpy()[rows]]

    # 4. Tune each model on the validation period (train on the training period only)
    results, tuning = [], []
    for name, (make, grid) in CANDIDATES.items():
        scored = []
        for params in grid:
            m = pipeline(make(params)).fit(X.iloc[train], y.iloc[train])
            scored.append((metrics(y.iloc[val], m.predict(X.iloc[val])), params, m))
        best = min(scored, key=lambda s: s[0]["MAE"])
        if len(grid) > 1:
            tuning.append({"model": name, "first_setting_MAE": scored[0][0]["MAE"],
                           "tuned_MAE": best[0]["MAE"], "best_params": best[1]})
        results.append({"model": name, "params": best[1], "validation": best[0], "fitted_on_train": best[2]})

    # 5. Select on validation, then retrain the winners on train + validation and test once
    selected = min(results, key=lambda r: r["validation"]["MAE"])
    finals = {}
    for r in results:
        finals[r["model"]] = pipeline(CANDIDATES[r["model"]][0](r["params"])).fit(X.iloc[train_val], y.iloc[train_val])
        r["test"] = metrics(y.iloc[test], finals[r["model"]].predict(X.iloc[test]))
    baseline = {"model": "Persistence (same as previous hour)", "params": {},
                "validation": metrics(y.iloc[val], X["lag_1"].iloc[val]),
                "test": metrics(y.iloc[test], X["lag_1"].iloc[test])}
    final = finals[selected["model"]]

    # 6. Flag threshold = 95th percentile of the selected model's validation errors (out-of-sample)
    val_err = np.abs(y.iloc[val].to_numpy() - selected["fitted_on_train"].predict(X.iloc[val]))
    threshold = float(np.quantile(val_err, 0.95))

    # 7. Predict the test period, flag large deviations, group them into episodes and classify them
    pred = pd.DataFrame({"actual": y.iloc[test], "predicted": final.predict(X.iloc[test])})
    pred["residual"] = pred["actual"] - pred["predicted"]
    pred["flagged"] = pred["residual"].abs() > threshold
    pred["missing_inputs"] = X.iloc[test].isna().sum(axis=1)
    missing_before = y.isna().rolling(6, min_periods=1).sum().shift(1)  # missing values in previous 6 h
    pred["missing_before"] = missing_before.reindex(pred.index).fillna(0).astype(int)

    episodes, current = [], []
    flagged_times = list(pred.index[pred["flagged"]])
    for t in flagged_times + [None]:
        if current and (t is None or t - current[-1] > pd.Timedelta(hours=3)):
            ep = pred.loc[current[0]:current[-1]]
            hits = ep[ep["flagged"]]
            reasons = []
            if hits["missing_inputs"].sum() > 0:
                reasons.append("some input readings were missing and had to be filled in")
            if hits["missing_before"].max() > 0:
                reasons.append(f"the pollutant data had a gap in the 6 hours before ({int(hits['missing_before'].max())} missing hours)")
            direction = "higher" if hits["residual"].mean() > 0 else "lower"
            if reasons:
                kind = "Data limitation"
                reason = "Probably a data problem: " + "; ".join(reasons) + "."
            else:
                kind = "Unusual pollution"
                reason = (f"All inputs were complete, yet the observed level was {direction} than expected "
                          f"by up to {hits['residual'].abs().max():.2f} {info['unit']}: a genuine unusual pollution event.")
            episodes.append({"id": len(episodes) + 1, "start": current[0].isoformat(), "end": current[-1].isoformat(),
                             "flagged_hours": int(len(hits)), "peak_actual": float(hits["actual"].max()),
                             "max_deviation": float(hits["residual"].abs().max()), "direction": direction,
                             "type": kind, "reason": reason})
            current = []
        if t is not None:
            current.append(t)

    # 8. Save the model (CPU inference, so it runs anywhere) and the results for the API/dashboard
    if selected["model"] == "XGBoost":
        final.named_steps["model"].set_params(device="cpu")
    joblib.dump(final, os.path.join(MODELS, f"{key}.pkl"))
    assert np.allclose(joblib.load(os.path.join(MODELS, f"{key}.pkl")).predict(X.iloc[test[:10]]), final.predict(X.iloc[test[:10]]))

    est = final.named_steps["model"]
    raw_imp = est.feature_importances_ if hasattr(est, "feature_importances_") else np.abs(est.coef_)
    importance = sorted(({"feature": f, "importance": float(v / raw_imp.sum())} for f, v in zip(FEATURES, raw_imp)),
                        key=lambda d: -d["importance"])
    s = y.dropna()
    sel_test, base_test = selected["test"]["MAE"], baseline["test"]["MAE"]
    out = {
        "key": key, **info, "features": FEATURES, "device_used": DEVICE,
        "selected_model": selected["model"], "selected_params": selected["params"], "threshold": threshold,
        "improvement_vs_persistence_pct": float((1 - sel_test / base_test) * 100),
        "periods": {"train": [df.index[24].isoformat(), df.index[train_end - 1].isoformat()],
                    "validation": [df.index[train_end].isoformat(), df.index[val_end - 1].isoformat()],
                    "test": [df.index[val_end].isoformat(), df.index[-1].isoformat()]},
        "models": [{k: v for k, v in r.items() if k != "fitted_on_train"} for r in results] + [baseline],
        "tuning": tuning, "feature_importance": importance,
        "data": {"observed_pct": float(y.notna().mean() * 100), "mean": float(s.mean()), "max": float(s.max()),
                 "daily": [[d.strftime("%Y-%m-%d"), None if np.isnan(v) else round(float(v), 3)]
                           for d, v in y.resample("D").mean().items()],
                 "hourly_profile": [round(float(v), 3) for v in y.groupby(df.index.hour).mean()]},
        "test_predictions": [{"t": t.isoformat(), "actual": float(r.actual), "predicted": round(float(r.predicted), 4),
                              "flagged": bool(r.flagged)} for t, r in pred.iterrows()],
        "episodes": episodes,
        "episode_counts": {"total": len(episodes),
                           "Unusual pollution": sum(e["type"] == "Unusual pollution" for e in episodes),
                           "Data limitation": sum(e["type"] == "Data limitation" for e in episodes)},
    }
    RESULTS[key] = out
    pred_frames[key] = pred
    print(f"{key:5s} {selected['model']:17s} test MAE {sel_test:8.3f} vs persistence {base_test:8.3f} "
          f"({out['improvement_vs_persistence_pct']:5.1f}% better) | episodes {out['episode_counts']}")

In [ ]:
print("XGBoost device:", DEVICE)
for pollutant in POLLUTANTS:
    train_pollutant(df, pollutant)

summary = pd.DataFrame([{
    "pollutant": k, "unit": r["unit"], "chosen model": r["selected_model"],
    "test MAE": next(m["test"]["MAE"] for m in r["models"] if m["model"] == r["selected_model"]),
    "persistence MAE": r["models"][-1]["test"]["MAE"],
    "better than persistence %": r["improvement_vs_persistence_pct"],
    "episodes": r["episode_counts"]["total"],
    "unusual pollution": r["episode_counts"]["Unusual pollution"],
    "data limitation": r["episode_counts"]["Data limitation"]} for k, r in RESULTS.items()])
summary.round(3)

## 6. Results for one pollutant

In [ ]:
SELECTED = "CO"   # change to "NO2", "NOx" or "C6H6"
r, pred = RESULTS[SELECTED], pred_frames[SELECTED]
pd.DataFrame([{"model": m["model"], "validation MAE": m["validation"]["MAE"], "test MAE": m["test"]["MAE"],
               "test R2": m["test"]["R2"]} for m in r["models"]]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.5))
week = pred.iloc[:24 * 21]                      # first three weeks of the test period
ax.plot(week.index, week["actual"], label="actual", lw=1)
ax.plot(week.index, week["predicted"], label="predicted", lw=1)
f = week[week["flagged"]]
ax.scatter(f.index, f["actual"], color="red", s=20, zorder=3, label="flagged")
ax.set_title(f"{r['name']} - actual vs predicted ({r['selected_model']})"); ax.set_ylabel(r["unit"]); ax.legend()
plt.show()
pd.DataFrame(r["episodes"])[["start", "end", "flagged_hours", "max_deviation", "type", "reason"]].head(15)